# Kaggle training launcher

This notebook is a launcher, not where the code lives. Everything it calls is
in the repo, version-controlled, linted and tested in CI. That keeps the real
work reviewable on GitHub instead of buried in cell outputs.

**Before the first run:**

1. Settings -> Accelerator -> **GPU T4 x2**. Use the T4, not the P100:
   bitsandbytes 4-bit needs compute capability 7.5+, and the P100 is 6.0.
   One of the two T4s is enough for a 2B model under QLoRA.
2. Settings -> **Internet: On**. Off by default; without it every model
   download and adapter push fails. Requires phone verification once.
3. Add-ons -> **Secrets** -> add `HF_TOKEN`. Never paste a token into a cell:
   `.ipynb` files keep their outputs, so a printed token survives into every
   later commit.
4. Run the smoke config first. It finishes in about five minutes and proves
   the loop works before you spend weekly quota on the real run.

Long runs go through **Save Version (commit-and-run)**, not an open browser tab —
interactive sessions idle out well before a fine-tune finishes.


## 1. Install


In [ ]:
!pip install -q git+https://github.com/sakshi-47/doc-extraction-cost-frontier.git

# Pulls docbench (the shared evaluation core) from its own public repo as a
# declared dependency. The metric is not duplicated here.


## 2. Check the hardware before committing to a config

Kaggle serves P100 (compute 6.0) and T4 (7.5). Both are pre-Ampere, so
`bfloat16` is unavailable and training runs in `float16` with gradient
scaling. Recipes written for an A100 will assume otherwise.


In [ ]:
!docbench hardware


## 3. Authenticate

Read from Kaggle Secrets. Nothing is printed and nothing is passed as an argument.


In [ ]:
from docbench.auth import get_token

hf = get_token('HF_TOKEN')
print(hf)  # redacted by __repr__ — safe to leave in a committed notebook


## 4. Smoke run — about five minutes

Validate the whole loop end to end. A run that diverges at hour three is a
wasted afternoon and a chunk of a weekly allowance.


In [ ]:
!docbench validate configs/smoke.yaml
!python -m docbench.train.loop --config configs/smoke.yaml


## 5. The real run

Checkpoints push to the Hub every 250 steps, because `/kaggle/working` does
not survive between sessions. If this session is killed, rerun the same cell
with `--resume` and it continues from the last pushed adapter.


In [ ]:
!python -m docbench.train.loop --config configs/qwen2vl_lora.yaml


In [ ]:
# Resume after an interrupted session:
# !python -m docbench.train.loop --config configs/qwen2vl_lora.yaml --resume
